# DTM Ground Classification — Iterative Self-Training Strategy
### Target: >95% Accuracy on Indian Village Terrain | T4 x2 GPU

## Why Previous Run Achieved Only 76%

| Problem | Evidence | Fix Applied |
|---|---|---|
| **CSF label ceiling** | train_acc ≈ val_acc ≈ 76% (model learned CSF noise) | Iterative pseudo-labeling breaks ceiling |
| **Double forward pass** | 170 min for 90 epochs (2× waste) | Cache logits, single forward per step |
| **Only 1 GPU used** | T4 x2 available, 31.2 GB total VRAM | DataParallel across both GPUs |
| **OneCycleLR broken** | Loss barely moved over 90 epochs | CosineAnnealingWarmRestarts |
| **No balanced sampling** | Ground 16%, model ignores minority | WeightedRandomSampler |

## Strategy: Iterative Pseudo-Labeling

```
Phase 1 [60 ep] — Train on CSF labels           → ~76%  (CSF ceiling)
Phase 2 [40 ep] — Refine: conf >= 0.80          → ~83%  (break ceiling)
Phase 3 [40 ep] — Refine: conf >= 0.87          → ~90%  (high quality labels)
Phase 4 [30 ep] — Refine: conf >= 0.92 + SWA    → ~95%+ (converge)
Phase 5         — F1-optimal threshold sweep     → final push
```

**KEY INSIGHT:** Model predictions in HIGH CONFIDENCE regions are MORE accurate
than CSF labels. Each round trains on progressively cleaner labels, breaking
the CSF accuracy ceiling that capped previous run at 76%.

**Estimated total time: ~4 hours out of 12-hour Kaggle budget**

| Cell | Role | Key names |
|---|---|---|
| 1 | GPU setup + DataParallel | `DEVICE`, `N_GPUS`, `SESSION_START` |
| 2 | Config | `CFG` |
| 3 | Feature engineering | `geo_features`, `build_feat_cache` |
| 4 | Model | `DTMPointNet2`, `model` |
| 5 | Dataset | `GroundDataset`, `make_loader` |
| 6 | Training engine | `run_epoch`, `train_phase` |
| 7 | Phase 1 — CSF labels | `history_p1`, `best_acc_p1` |
| 8 | Iterative pseudo-labeling | `refine_labels`, phases 2-4 |
| 9 | Threshold sweep + package | `dtm_outputs.zip` |


In [ ]:
# Cell 1 — GPU detection, DataParallel setup, seed
import torch, torch.nn as nn, random, time, os
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU.\nKaggle -> Settings -> Accelerator -> GPU T4 x2 -> Save\n'
        'Run -> Restart Session -> Run All Cells'
    )

N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda')

print(f'GPUs detected : {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    sm = p.major*10+p.minor
    print(f'  GPU {i}: {p.name}  sm_{sm}  {p.total_memory/1e9:.1f} GB')
    if sm < 70:
        raise RuntimeError(f'GPU {i} (sm_{sm}) too old. Need T4 (sm_75)+')

# Probe both GPUs
for i in range(N_GPUS):
    _t = torch.zeros(4, device=f'cuda:{i}') + 1
    torch.cuda.synchronize(i)
    del _t
print('All GPU probes passed')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

SESSION_START    = time.time()
SESSION_BUDGET_H = 11.0

print(f'DataParallel : {N_GPUS} GPU(s)')
print(f'Total VRAM   : {sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPUS))/1e9:.1f} GB')
print(f'Session clock: started (budget={SESSION_BUDGET_H}h)')


In [ ]:
# Cell 2 — Configuration
from pathlib import Path

DATASET_ROOT = '/kaggle/input/datasets/jaideepchouhan/training-data-95pct'

def _find_split_root(base):
    base = Path(base)
    if (base/'train').exists() and (base/'val').exists(): return str(base), None
    for sub in sorted(base.iterdir()):
        if sub.is_dir() and (sub/'train').exists(): return str(sub), None
    zips = list(base.rglob('training_data.zip'))
    if zips: return None, str(zips[0])
    return str(base), None

_TDIR, _ZPATH = _find_split_root(DATASET_ROOT)
_USE_ZIP = _ZPATH is not None and _TDIR is None

# Batch per GPU; total effective batch = BATCH_PER_GPU * N_GPUS
BATCH_PER_GPU = 8
TOTAL_BATCH   = BATCH_PER_GPU * N_GPUS   # 16 with T4 x2

CFG = {
    'use_zip'          : _USE_ZIP,
    'zip_path'         : _ZPATH or '',
    'training_dir'     : _TDIR  or '',
    'feat_cache_dir'   : '/kaggle/working/feat_cache',
    'refined_dir'      : '/kaggle/working/refined_labels',
    'logs_dir'         : '/kaggle/working/logs',
    'model_save_path'  : '/kaggle/working/logs/best_model.pth',
    'swa_save_path'    : '/kaggle/working/logs/swa_model.pth',
    'ckpt_path'        : '/kaggle/working/logs/checkpoint.pth',
    'history_path'     : '/kaggle/working/logs/history.json',
    'curves_path'      : '/kaggle/working/logs/training_curves.png',
    'threshold_path'   : '/kaggle/working/logs/optimal_threshold.json',
    'max_pts'          : 4096,
    'seed'             : 42,
    'batch_size'       : TOTAL_BATCH,
    'max_train_tiles'  : 99999,
    'max_val_tiles'    : 99999,
    # Phase 1 — CSF labels
    'phase1_epochs'    : 60,
    'phase1_lr'        : 3e-3,
    # Phase 2-4 — Iterative refinement
    'refine_epochs'    : [40, 35, 30],
    'refine_conf'      : [0.80, 0.87, 0.92],
    'refine_lr'        : 1e-3,
    'refine_tta'       : 5,       # test-time augmentation passes for pseudo-labels
    # Optimiser
    'weight_decay'     : 1e-4,
    'grad_clip'        : 3.0,
    # Focal Loss
    'focal_alpha'      : 0.75,
    'focal_gamma'      : 2.0,
    'label_smooth'     : 0.05,
    # SWA (starts in final phase)
    'swa_lr'           : 5e-4,
    # Training
    'use_amp'          : True,
    'val_every'        : 2,
    'patience'         : 15,
    'num_workers'      : 4,
    'prefetch_factor'  : 2,
}

for d in [CFG['logs_dir'], CFG['feat_cache_dir'], CFG['refined_dir']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Data source  : {"ZIP:"+str(_ZPATH) if _USE_ZIP else _TDIR}')
print(f'Batch/GPU    : {BATCH_PER_GPU}  |  Total batch: {TOTAL_BATCH}')
print(f'Phase 1      : {CFG["phase1_epochs"]} epochs')
print(f'Refine rounds: {len(CFG["refine_epochs"])} x epochs {CFG["refine_epochs"]}')
print(f'Conf thresh  : {CFG["refine_conf"]}')


In [ ]:
# Cell 3 — Feature Engineering + Cache
# 6 terrain features (upgraded from 4):
#   dZ_2m     = height above 2m-grid local min (0=ground, >0=object)
#   roughness = Z std-dev in 2m cell
#   slope     = max gradient across neighbours / 2m
#   density   = normalised point count
#   dZ_8m     = height above 8m-grid local min (captures tall trees/buildings)
#   planarity = std(Z) / range(Z) in cell (flat=ground, irregular=veg)
# Input: (N,3) -> Output: (N,6) z-scored

import numpy as np
from pathlib import Path, PurePosixPath
from tqdm import tqdm
import time, io, zipfile


def _grid_stats(xyz, grid_m, GW_max=64):
    GW = int(np.clip((xyz[:,0].max()-xyz[:,0].min()+1e-6)/grid_m, 8, GW_max))
    GH = int(np.clip((xyz[:,1].max()-xyz[:,1].min()+1e-6)/grid_m, 8, GW_max))
    xmin,ymin = xyz[:,0].min(), xyz[:,1].min()
    xrng = xyz[:,0].max()-xmin+1e-6
    yrng = xyz[:,1].max()-ymin+1e-6
    xi = np.clip(((xyz[:,0]-xmin)/xrng*GW).astype(np.int32), 0, GW-1)
    yi = np.clip(((xyz[:,1]-ymin)/yrng*GH).astype(np.int32), 0, GH-1)
    ci = yi*GW+xi; NC=GW*GH; z=xyz[:,2].astype(np.float32)
    c_min=np.full(NC,np.inf,np.float32)
    c_sum=np.zeros(NC,np.float32); c_sq=np.zeros(NC,np.float32)
    c_cnt=np.zeros(NC,np.float32); c_max=np.full(NC,-np.inf,np.float32)
    np.minimum.at(c_min,ci,z); np.maximum.at(c_max,ci,z)
    np.add.at(c_sum,ci,z); np.add.at(c_sq,ci,z*z); np.add.at(c_cnt,ci,1.)
    empty=c_cnt==0
    c_cnt[empty]=1; c_min[empty]=z.mean(); c_max[empty]=z.mean()
    c_sum[empty]=z.mean(); c_sq[empty]=z.mean()**2
    c_mean=c_sum/c_cnt
    c_std=np.sqrt(np.maximum(c_sq/c_cnt-c_mean**2,0.))
    c_rng=c_max-c_min
    return ci, c_min, c_std, c_cnt, c_rng, GW, GH


def geo_features(xyz: np.ndarray) -> np.ndarray:
    """
    (N,3) float32 -> (N,6) float32 z-scored terrain features.
    6 features: dZ_2m, roughness, slope, density, dZ_8m, planarity
    """
    xyz = xyz.astype(np.float32, copy=False)

    # 2m grid stats
    ci2, c_min2, c_std2, c_cnt2, c_rng2, GW2, GH2 = _grid_stats(xyz, 2.0)

    # dZ_2m: height above 2m-grid local minimum
    dz2 = np.clip(xyz[:,2]-c_min2[ci2], 0., None)

    # roughness: Z std in 2m cell
    roughness = c_std2[ci2]

    # slope: max |gradient| across 4 neighbours in 2m grid
    cg = c_min2.reshape(GH2, GW2)
    pad = np.pad(cg, 1, mode='edge')
    sg = np.max(np.stack([
        np.abs(cg-pad[:-2,1:-1]), np.abs(cg-pad[2:,1:-1]),
        np.abs(cg-pad[1:-1,:-2]), np.abs(cg-pad[1:-1,2:]),
    ]),axis=0)/2.0
    slope = sg.reshape(-1)[ci2]

    # density: normalised point count per 2m cell
    density = c_cnt2[ci2]/(c_cnt2.max()+1e-6)

    # 8m grid: dZ_8m — captures tall trees and buildings
    ci8, c_min8, _, _, _, _, _ = _grid_stats(xyz, 8.0, GW_max=32)
    dz8 = np.clip(xyz[:,2]-c_min8[ci8], 0., None)

    # planarity: std/range in 2m cell (0=flat ground, 1=rough veg)
    planarity = c_std2[ci2] / (c_rng2[ci2]+1e-6)

    feat = np.stack([dz2,roughness,slope,density,dz8,planarity],axis=1)
    mu  = feat.mean(axis=0,keepdims=True)
    sig = feat.std(axis=0,keepdims=True)+1e-6
    return ((feat-mu)/sig).astype(np.float32)


def build_feat_cache(cfg: dict, split: str):
    cache_dir = Path(cfg['feat_cache_dir'])/split
    cache_dir.mkdir(parents=True, exist_ok=True)
    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tiles={}
        for n in names:
            parts=PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tiles[parts[parts.index(split)+1]]=n
        zf_h=zipfile.ZipFile(cfg['zip_path'])
        try:
            for tile,zp in tqdm(tiles.items(),desc=f'Cache[{split}]',unit='tile'):
                out=cache_dir/f'{tile}.npy'
                if out.exists(): continue
                with zf_h.open(zp) as fh: xyz=np.load(io.BytesIO(fh.read()))
                np.save(str(out),geo_features(xyz))
        finally: zf_h.close()
    else:
        for td in tqdm(sorted((Path(cfg['training_dir'])/split).glob('tile_*')),
                       desc=f'Cache[{split}]',unit='tile'):
            out=cache_dir/f'{td.name}.npy'
            if out.exists(): continue
            np.save(str(out),geo_features(np.load(str(td/'points.npy'))))


print('Building feature caches...')
t0=time.time()
build_feat_cache(CFG,'train'); build_feat_cache(CFG,'val')
print(f'Cache done in {(time.time()-t0)/60:.1f} min')
print('geo_features: 6-channel output [dZ_2m, roughness, slope, density, dZ_8m, planarity]')


In [ ]:
# Cell 4 — DTMPointNet2 (9-channel input: 3 XYZ + 6 terrain features)
# Upgraded: larger channels, residual head, DataParallel wrapper

import torch, torch.nn as nn, torch.nn.functional as F

# ── Geometric ops ────────────────────────────────────────────────────────────
def _fps(xyz,n):
    B,N,_=xyz.shape; dev=xyz.device
    cent=torch.zeros(B,n,dtype=torch.long,device=dev)
    dist=torch.full((B,N),1e10,dtype=torch.float32,device=dev)
    far=torch.randint(0,N,(B,),dtype=torch.long,device=dev)
    bi=torch.arange(B,dtype=torch.long,device=dev)
    for i in range(n):
        cent[:,i]=far
        c=xyz[bi,far].unsqueeze(1)
        d=((xyz-c)**2).sum(-1)
        dist=torch.where(d<dist,d,dist)
        far=dist.argmax(-1)
    return cent

def _idx(pts,idx):
    B=pts.shape[0]
    bi=torch.arange(B,device=pts.device).view([B]+[1]*(idx.dim()-1)).expand_as(idx)
    return pts[bi,idx]

def _ball(new_xyz,xyz,r,k):
    d=torch.cdist(new_xyz,xyz)
    m=torch.where(d<=r,d,d.new_full((),1e10))
    return m.topk(k,dim=-1,largest=False).indices

# ── Building blocks ───────────────────────────────────────────────────────────
class _MSG(nn.Module):
    def __init__(self,n_ctr,radii,nsamps,in_ch,specs):
        super().__init__()
        self.n_ctr=n_ctr; self.radii=radii; self.nsamps=nsamps
        self.mlps=nn.ModuleList()
        for dims in specs:
            layers,ch=[],in_ch+3
            for d in dims:
                layers+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]
                ch=d
            self.mlps.append(nn.Sequential(*layers))
    def forward(self,xyz,feat):
        ci=_fps(xyz,self.n_ctr); nxyz=_idx(xyz,ci); outs=[]
        for r,k,mlp in zip(self.radii,self.nsamps,self.mlps):
            idx=_ball(nxyz,xyz,r,k)
            grp=_idx(feat,idx)
            rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
            grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
            outs.append(mlp(grp).max(2).values)
        return nxyz,torch.cat(outs,1).permute(0,2,1)

class _SA(nn.Module):
    def __init__(self,r,k,in_ch,dims):
        super().__init__(); self.r=r; self.k=k
        layers,ch=[],in_ch+3
        for d in dims:
            layers+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*layers)
    def forward(self,xyz,feat):
        n_ctr=max(1,xyz.shape[1]//4); ci=_fps(xyz,n_ctr); nxyz=_idx(xyz,ci)
        idx=_ball(nxyz,xyz,self.r,self.k); grp=_idx(feat,idx)
        rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
        grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
        return nxyz,self.mlp(grp).max(2).values.permute(0,2,1)

class _FP(nn.Module):
    def __init__(self,in_ch,dims):
        super().__init__()
        layers,ch=[],in_ch
        for d in dims:
            layers+=[nn.Conv1d(ch,d,1,bias=False),nn.BatchNorm1d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*layers)
    def forward(self,xyz1,xyz2,f1,f2):
        d=torch.cdist(xyz1,xyz2); t3=d.topk(3,dim=-1,largest=False)
        w=1./(t3.values+1e-8); w=w/w.sum(-1,keepdim=True)
        interp=(_idx(f2,t3.indices)*w.unsqueeze(-1)).sum(2)
        feat=torch.cat([f1,interp],-1) if f1 is not None else interp
        return self.mlp(feat.permute(0,2,1)).permute(0,2,1)

# ── Full model (9-channel input) ──────────────────────────────────────────────
class DTMPointNet2(nn.Module):
    """
    Input : (B, N, 9)  [x,y,z, dZ_2m, roughness, slope, density, dZ_8m, planarity]
    Output: (B, N, 2)  logits [non-ground, ground]
    Channels upgraded: SA1->256, SA2->512, SA3->1024
    """
    def __init__(self, in_feat: int = 9):
        super().__init__()
        C = in_feat
        # SA1: 0.5m=fine kuccha wall texture, 1.5m=compound boundary context
        self.sa1 = _MSG(512,[0.5,1.5],[32,64],C,
                        [[64,64],[64,128]])          # -> 192-ch
        # SA2: 3m=building footprint, 8m=compound block
        self.sa2 = _MSG(128,[3.0,8.0],[64,128],192,
                        [[128,192],[128,256]])        # -> 448-ch
        # SA3: global context
        self.sa3 = _SA(15.0,128,448,[256,512])       # -> 512-ch
        self.fp3 = _FP(512+448,[256,256])
        self.fp2 = _FP(256+192,[256,128])
        self.fp1 = _FP(128+C,  [128,128,64])
        self.head = nn.Sequential(
            nn.Conv1d(64,64,1,bias=False), nn.BatchNorm1d(64), nn.ReLU(True),
            nn.Dropout(0.4),
            nn.Conv1d(64,2,1),
        )
    def forward(self,pts):
        xyz0=pts[:,:,:3].contiguous(); f0=pts.contiguous()
        xyz1,f1=self.sa1(xyz0,f0)
        xyz2,f2=self.sa2(xyz1,f1)
        xyz3,f3=self.sa3(xyz2,f2)
        f2=self.fp3(xyz2,xyz3,f2,f3)
        f1=self.fp2(xyz1,xyz2,f1,f2)
        f0=self.fp1(xyz0,xyz1,f0,f1)
        return self.head(f0.permute(0,2,1)).permute(0,2,1)  # (B,N,2)


IN_FEAT = 9  # 3 XYZ + 6 terrain
_base   = DTMPointNet2(IN_FEAT).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(_base)
    print(f'DataParallel across {N_GPUS} GPUs')
else:
    model = _base

# Sanity check
_x=torch.randn(2,128,IN_FEAT,device=DEVICE)
with torch.no_grad(): _o=model(_x)
assert _o.shape==(2,128,2), f'Bad shape: {_o.shape}'
del _x,_o; torch.cuda.empty_cache()

n_p=sum(p.numel() for p in _base.parameters())
print(f'Model: DTMPointNet2  |  {n_p/1e6:.2f} M params  |  in_feat={IN_FEAT}')
print('Cell 4 done')


In [ ]:
# Cell 5 — Dataset + WeightedRandomSampler
import io, zipfile
import numpy as np, torch
from pathlib import Path, PurePosixPath
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class GroundDataset(Dataset):
    """
    Returns (N,9) tensor [xyz + 6 geo_features].
    `label_dir` overrides the default labels.npy with refined labels from
    iterative pseudo-labeling.
    """
    def __init__(self, cfg, split='train', augment=False,
                 max_tiles=None, label_dir=None):
        self.augment  = augment
        self.max_pts  = cfg['max_pts']
        self.seed     = cfg['seed']
        self.split    = split
        self.use_zip  = cfg['use_zip']
        self.cache    = Path(cfg['feat_cache_dir'])/split
        self.lbl_dir  = Path(label_dir) if label_dir else None

        if self.use_zip:
            self._zpath=cfg['zip_path']; self._tiles=self._disc_zip()
        else:
            self._tdir=Path(cfg['training_dir'])/split
            self._tiles=sorted(p.name for p in self._tdir.glob('tile_*')
                               if (p/'labels.npy').exists())

        if max_tiles and len(self._tiles)>max_tiles:
            rng=np.random.default_rng(self.seed)
            pick=rng.choice(len(self._tiles),max_tiles,replace=False)
            self._tiles=[self._tiles[i] for i in sorted(pick)]

        print(f'  [{split}] {len(self._tiles)} tiles | lbl_dir={label_dir is not None}')

    def _disc_zip(self):
        with zipfile.ZipFile(self._zpath) as zf: names=zf.namelist()
        t=set()
        for n in names:
            parts=PurePosixPath(n).parts
            if self.split in parts and 'tile_' in parts[-2]:
                t.add(parts[parts.index(self.split)+1])
        return sorted(t)

    def _load_zip(self,tile):
        if not hasattr(self,'_zf') or self._zf is None:
            self._zf=zipfile.ZipFile(self._zpath)
        def rd(fn):
            with self._zf.open(f'{self.split}/{tile}/{fn}') as fh:
                return np.load(io.BytesIO(fh.read()))
        return rd('points.npy'),rd('labels.npy')

    def __len__(self): return len(self._tiles)

    def __getitem__(self,idx):
        tile=self._tiles[idx]
        rng=np.random.default_rng(idx+self.seed*10000)

        if self.use_zip: xyz,labels=self._load_zip(tile)
        else:
            xyz=np.load(str(self._tdir/tile/'points.npy'))
            labels=np.load(str(self._tdir/tile/'labels.npy'))

        # Override with refined labels if available
        if self.lbl_dir is not None:
            p=self.lbl_dir/f'{tile}.npy'
            if p.exists(): labels=np.load(str(p))

        if self.augment:
            t=rng.uniform(0,2*np.pi)
            R=np.array([[np.cos(t),-np.sin(t)],[np.sin(t),np.cos(t)]],np.float32)
            xyz[:,:2]=xyz[:,:2]@R.T
            xyz[:,2]+=rng.normal(0,0.02,len(xyz)).astype(np.float32)
            xyz*=rng.uniform(0.93,1.07)
            if rng.random()>0.5: xyz[:,0]*=-1
            if rng.random()>0.5: xyz[:,1]*=-1
            xyz[:,0]*=rng.uniform(0.88,1.12); xyz[:,1]*=rng.uniform(0.88,1.12)
            xyz[:,2]*=rng.uniform(0.88,1.12)
            keep=rng.random(len(xyz))>0.10
            if keep.sum()>64: xyz=xyz[keep]; labels=labels[keep]

        N=len(xyz)
        if N>self.max_pts:
            ch=rng.choice(N,self.max_pts,replace=False); xyz=xyz[ch]; labels=labels[ch]
        elif N<self.max_pts:
            pad=rng.choice(N,self.max_pts-N,replace=True)
            xyz=np.concatenate([xyz,xyz[pad]]); labels=np.concatenate([labels,labels[pad]])

        cp=self.cache/f'{tile}.npy'
        feat6=(np.load(str(cp)) if cp.exists() and not self.augment
               else geo_features(xyz))
        if len(feat6)!=self.max_pts: feat6=geo_features(xyz)

        xn=xyz.copy(); xn[:,:2]-=xn[:,:2].mean(0)
        xn[:,2]-=float(np.median(xn[:,2])); xn/=(np.abs(xn).max()+1e-6)

        pts9=np.concatenate([xn,feat6],axis=1).astype(np.float32)
        return torch.from_numpy(pts9),torch.from_numpy(labels.astype(np.int64))


def make_loader(cfg, split, augment=False, label_dir=None,
                max_tiles=None, use_sampler=False):
    ds = GroundDataset(cfg, split, augment=augment,
                       max_tiles=max_tiles, label_dir=label_dir)
    nw = cfg['num_workers']
    kw = dict(num_workers=nw, pin_memory=True,
              persistent_workers=(nw>0),
              prefetch_factor=cfg['prefetch_factor'] if nw>0 else None)

    if use_sampler and split == 'train':
        # WeightedRandomSampler: each batch balanced ~50/50 ground/non-ground
        weights = []
        for tile in ds._tiles:
            if ds.use_zip:
                _,lbl=ds._load_zip(tile)
            else:
                lbl=np.load(str(ds._tdir/tile/'labels.npy'))
            gf=lbl.mean()
            # Weight tile inversely proportional to its majority class
            w=1.0/(gf+0.05) if gf<0.5 else 1.0/((1-gf)+0.05)
            weights.append(float(w))
        sampler=WeightedRandomSampler(weights,len(weights),replacement=True)
        return DataLoader(ds, batch_size=cfg['batch_size'],
                          sampler=sampler, drop_last=True, **kw)

    shuffle=(split=='train')
    return DataLoader(ds, batch_size=cfg['batch_size'],
                      shuffle=shuffle, drop_last=shuffle, **kw)


# Smoke test
_ds=GroundDataset(CFG,'train',augment=False,max_tiles=2)
_p,_l=_ds[0]
assert _p.shape==(CFG['max_pts'],9), f'Expected (4096,9) got {_p.shape}'
del _ds,_p,_l
print(f'GroundDataset smoke test PASSED: shape=({CFG["max_pts"]},9)')
print('Cell 5 done')


In [ ]:
# Cell 6 — Training Engine (reusable across all phases)
# Fixes from v4:
#   1. logits cached — model called ONCE per step (was 2x)
#   2. CosineAnnealingWarmRestarts replaces broken OneCycleLR
#   3. DataParallel aware
#   4. WeightedRandomSampler for balanced batches

import contextlib, time, json, math
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# Dependency check
_NEED={'CFG':'Cell2','DEVICE':'Cell1','SESSION_START':'Cell1',
       'geo_features':'Cell3','DTMPointNet2':'Cell4',
       'model':'Cell4','GroundDataset':'Cell5','make_loader':'Cell5'}
_miss={n:c for n,c in _NEED.items() if n not in dir()}
if _miss:
    raise NameError('Missing: '+str(_miss)+'\nRun -> Run All Cells')
print('Dependencies OK')

# AMP shim
def _scaler(on):
    try: return torch.amp.GradScaler('cuda',enabled=on)
    except: return torch.cuda.amp.GradScaler(enabled=on)

def _autocast(on):
    try: return torch.amp.autocast('cuda',enabled=on)
    except: return torch.cuda.amp.autocast(enabled=on)


class FocalLoss(nn.Module):
    def __init__(self,alpha,gamma,smooth):
        super().__init__(); self.a=alpha; self.g=gamma; self.s=smooth
    def forward(self,logits,targets):
        nc=logits.size(-1)
        soft=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1.)
        soft=soft*(1-self.s)+self.s/nc
        ce=-(soft*torch.log_softmax(logits,-1)).sum(-1)
        pt=torch.exp(-ce)
        at=torch.where(targets==1,
            targets.new_full(targets.shape,self.a,dtype=torch.float32),
            targets.new_full(targets.shape,1-self.a,dtype=torch.float32))
        return (at*(1-pt)**self.g*ce).mean()


def _metrics(preds,labels):
    acc=(preds==labels).float().mean().item()
    tp=((preds==1)&(labels==1)).sum().item()
    fp=((preds==1)&(labels==0)).sum().item()
    fn=((preds==0)&(labels==1)).sum().item()
    p=tp/(tp+fp+1e-9); r=tp/(tp+fn+1e-9)
    return acc, 2*p*r/(p+r+1e-9)


def train_phase(cfg, model, phase_name,
                n_epochs, lr, label_dir=None,
                use_swa=False, resume_ckpt=None):
    """
    Single training phase. Returns (history, best_acc).
    label_dir: if set, uses refined labels instead of original CSF labels.
    use_swa: enable SWA averaging in final third of training.
    """
    device=DEVICE
    crit=FocalLoss(cfg['focal_alpha'],cfg['focal_gamma'],cfg['label_smooth'])

    base_model=model.module if hasattr(model,'module') else model
    opt=torch.optim.AdamW(base_model.parameters(),
                          lr=lr/10., weight_decay=cfg['weight_decay'])

    tr_ld=make_loader(cfg,'train',augment=True,label_dir=label_dir,use_sampler=True)
    va_ld=make_loader(cfg,'val',  augment=False)

    # CosineAnnealingWarmRestarts: T_0=first restart period
    # Warms up to lr in T_0 epochs, then restarts with decay
    T0=max(10,n_epochs//3)
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=T0, T_mult=2, eta_min=lr*0.01)
    # Linear warmup for first 5 epochs
    warmup=torch.optim.lr_scheduler.LinearLR(
        opt, start_factor=0.1, end_factor=1.0, total_iters=5)
    sched=torch.optim.lr_scheduler.SequentialLR(
        opt,[warmup,sched],milestones=[5])

    swa_m=AveragedModel(base_model) if use_swa else None
    swa_s=SWALR(opt,swa_lr=cfg['swa_lr']) if use_swa else None
    swa_start=max(1,int(n_epochs*0.65)) if use_swa else n_epochs+1

    scaler=_scaler(cfg['use_amp'])
    history=[]; best_acc=0.; pat=0; start_ep=1
    t0=time.time()

    if resume_ckpt and Path(resume_ckpt).exists():
        try:
            c=torch.load(resume_ckpt,map_location=device)
            base_model.load_state_dict(c['model'])
            opt.load_state_dict(c['opt'])
            scaler.load_state_dict(c['scaler'])
            history=c.get('history',[]); best_acc=c.get('best_acc',0.)
            pat=c.get('pat',0); start_ep=c['epoch']+1
            print(f'Resumed {phase_name} from epoch {c["epoch"]} best={best_acc:.4f}')
        except Exception as e: print(f'Resume failed: {e}')

    print(f'\n{"="*60}')
    print(f'  {phase_name}  eps={start_ep}->{n_epochs}  lr={lr}')
    print(f'  batches/ep={len(tr_ld)}  lbl_dir={label_dir is not None}')
    print(f'{"="*60}\n')

    for ep in range(start_ep, n_epochs+1):
        if (time.time()-SESSION_START)/3600>=SESSION_BUDGET_H:
            print(f'Budget reached at ep {ep}'); break

        model.train()
        tr_loss=0.; tr_p=[]; tr_l=[]

        for pts,lbs in tqdm(tr_ld,desc=f'[{phase_name}] Ep{ep:3d} train',
                            leave=False,ncols=90):
            pts=pts.to(device,non_blocking=True)
            lbs=lbs.to(device,non_blocking=True)
            opt.zero_grad()
            with _autocast(cfg['use_amp']):
                logits=model(pts)                          # (B,N,2)
                loss=crit(logits.view(-1,2),lbs.view(-1)) # scalar
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(base_model.parameters(),cfg['grad_clip'])
            scaler.step(opt); scaler.update()
            # FIX: reuse logits — no second forward pass
            with torch.no_grad():
                tr_p.append(logits.detach().argmax(-1).view(-1).cpu())
                tr_l.append(lbs.view(-1).cpu())
            tr_loss+=loss.item()

        sched.step()
        ta,tf1=_metrics(torch.cat(tr_p),torch.cat(tr_l))
        tr_loss/=len(tr_ld)

        do_val=(ep%cfg['val_every']==0) or ep==n_epochs
        va=float('nan'); vf1=float('nan')

        if do_val:
            model.eval()
            vp=[]; vl=[]
            with torch.no_grad():
                for pts,lbs in tqdm(va_ld,desc=f'[{phase_name}] Ep{ep:3d} val',
                                    leave=False,ncols=90):
                    pts=pts.to(device,non_blocking=True)
                    with _autocast(cfg['use_amp']): out=model(pts)
                    vp.append(out.argmax(-1).view(-1).cpu())
                    vl.append(lbs.view(-1).cpu())
            va,vf1=_metrics(torch.cat(vp),torch.cat(vl))
            if va>best_acc:
                best_acc=va; pat=0
                torch.save(base_model.state_dict(),cfg['model_save_path'])
            else: pat+=1

        if use_swa and ep>=swa_start:
            swa_m.update_parameters(base_model); swa_s.step()

        star=' BEST' if (do_val and va==best_acc) else ''
        print(f'Ep{ep:3d} tl={tr_loss:.4f} ta={ta:.4f} tf1={tf1:.4f}'
              f' | va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')

        history.append(dict(ep=ep,phase=phase_name,tl=tr_loss,
                            ta=ta,tf1=tf1,va=va,vf1=vf1))
        torch.save(dict(epoch=ep,model=base_model.state_dict(),
                        opt=opt.state_dict(),scaler=scaler.state_dict(),
                        history=history,best_acc=best_acc,pat=pat),
                   cfg['ckpt_path'])

        if pat>=cfg['patience']: print(f'Early stop ep {ep}'); break

    if use_swa and swa_m is not None:
        print('Updating SWA BN...')
        bn_ld=make_loader(cfg,'train',augment=False,label_dir=label_dir)
        torch.optim.swa_utils.update_bn(bn_ld,swa_m,device=device)
        torch.save(swa_m.module.state_dict(),cfg['swa_save_path'])
        print(f'SWA saved')

    elapsed=(time.time()-t0)/60
    print(f'\n{phase_name} done in {elapsed:.1f} min  |  best_acc={best_acc:.4f}')
    return history,best_acc


def plot_all_history(all_history, path):
    fig,ax=plt.subplots(1,2,figsize=(13,4))
    colors={'Phase1':'#4a90d9','Phase2':'#e8a84c','Phase3':'#c4563a','Phase4':'#3dd6b5'}
    for h in all_history:
        ph=h.get('phase','?'); c=colors.get(ph,'gray')
        ax[0].scatter(h['ep'],h['tl'],c=c,s=10,alpha=0.7)
        if not math.isnan(h['va']):
            ax[1].scatter(h['ep'],h['va'],c=c,s=20,alpha=0.9,label=ph)
    ax[0].set(title='Focal Loss',xlabel='Epoch'); ax[0].grid(alpha=0.3)
    ax[1].axhline(0.95,ls='--',c='red',alpha=0.5,label='95% target')
    ax[1].set(title='Val Accuracy',xlabel='Epoch',ylim=(0.7,1.0))
    handles,labels=ax[1].get_legend_handles_labels()
    seen=set(); dedup=[]
    for h,l in zip(handles,labels):
        if l not in seen: dedup.append((h,l)); seen.add(l)
    ax[1].legend(*zip(*dedup) if dedup else ([],[]))
    ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path,dpi=120); plt.close()
    print(f'Curves: {path}')


print('Training engine ready. Cell 6 done.')


In [ ]:
# Cell 7 — Phase 1: Train on CSF labels
# Expected outcome: ~75-77% (CSF ceiling)
# This phase establishes a base model good enough to generate
# pseudo-labels that are MORE accurate than CSF in high-confidence regions.

ALL_HISTORY = []

print('PHASE 1: Training on CSF labels')
print(f'  Epochs: {CFG["phase1_epochs"]}')
print(f'  LR    : {CFG["phase1_lr"]}')
print(f'  Goal  : establish base model (~75-77%)')
print()

history_p1, best_acc_p1 = train_phase(
    cfg        = CFG,
    model      = model,
    phase_name = 'Phase1',
    n_epochs   = CFG['phase1_epochs'],
    lr         = CFG['phase1_lr'],
    label_dir  = None,           # use original CSF labels
    use_swa    = False,           # no SWA yet
    resume_ckpt= CFG['ckpt_path'],
)
ALL_HISTORY.extend(history_p1)

print(f'\nPhase 1 complete: best_acc={best_acc_p1*100:.2f}%')
print('Loading best Phase 1 weights for pseudo-labeling...')

_base_model = model.module if hasattr(model,'module') else model
_base_model.load_state_dict(
    torch.load(CFG['model_save_path'], map_location=DEVICE))
print('Phase 1 weights loaded. Ready for Cell 8 (iterative refinement).')


In [ ]:
# Cell 8 — Iterative Pseudo-Labeling (the key to breaking 95%)
#
# ALGORITHM:
# For each refinement round r with confidence threshold T_r:
#   1. Run model on ALL training tiles
#   2. For each point in each tile:
#      if max(softmax(logits)) >= T_r:  label = model_prediction  (more accurate than CSF)
#      else:                             label = original_CSF_label
#   3. Save refined labels as /kaggle/working/refined_labels/<round>/<tile>.npy
#   4. Retrain model on these hybrid labels
#
# WHY IT WORKS:
#   A 76% accurate model, when it predicts with 90%+ confidence,
#   is right ~90%+ of the time on those specific points.
#   These confident predictions are MORE reliable than CSF labels (~76%).
#   Each round improves label quality -> model quality -> better pseudo-labels.

import torch.nn.functional as F
import numpy as np
from pathlib import Path
from tqdm import tqdm
import io, zipfile


def refine_labels(model, cfg, device, conf_thresh, tta_passes, round_name):
    """
    Generate refined labels for all training tiles.
    Uses test-time augmentation (TTA): runs model tta_passes times with
    different random subsamples and averages probabilities.
    More TTA passes -> more reliable pseudo-labels.

    Returns: Path to directory containing refined label .npy files.
    """
    base_m = model.module if hasattr(model,'module') else model
    base_m.eval()

    out_dir = Path(cfg['refined_dir']) / round_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Discover training tiles
    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf:
            names = zf.namelist()
        tile_names = set()
        for n in names:
            from pathlib import PurePosixPath
            parts = PurePosixPath(n).parts
            if 'train' in parts and 'tile_' in parts[-2]:
                tile_names.add(parts[parts.index('train')+1])
        tile_names = sorted(tile_names)
        use_zip = True
    else:
        tdir = Path(cfg['training_dir'])/'train'
        tile_names = sorted(p.name for p in tdir.glob('tile_*')
                            if (p/'labels.npy').exists())
        use_zip = False

    max_pts = cfg['max_pts']
    n_replaced = 0
    n_total_pts = 0
    rng = np.random.default_rng(cfg['seed'])

    zf_h = zipfile.ZipFile(cfg['zip_path']) if use_zip else None
    cache_dir = Path(cfg['feat_cache_dir'])/'train'

    try:
        for tile in tqdm(tile_names, desc=f'Refine [{round_name}]', unit='tile'):

            # Load tile
            if use_zip:
                def _rd(fn):
                    with zf_h.open(f'train/{tile}/{fn}') as fh:
                        return np.load(io.BytesIO(fh.read()))
                xyz   = _rd('points.npy')
                orig_labels = _rd('labels.npy')
            else:
                xyz   = np.load(str(tdir/tile/'points.npy'))
                orig_labels = np.load(str(tdir/tile/'labels.npy'))

            N = len(xyz)
            # Accumulate probabilities across TTA passes
            avg_probs = np.zeros(N, dtype=np.float64)  # P(ground)

            for _ in range(tta_passes):
                # Random subsample to max_pts
                if N > max_pts:
                    idx = rng.choice(N, max_pts, replace=False)
                    sub_xyz = xyz[idx]
                else:
                    idx = np.arange(N)
                    sub_xyz = xyz

                feat6 = geo_features(sub_xyz)
                xn = sub_xyz.copy()
                xn[:,:2] -= xn[:,:2].mean(0)
                xn[:,2]  -= float(np.median(xn[:,2]))
                xn /= (np.abs(xn).max()+1e-6)

                pts9 = np.concatenate([xn,feat6],axis=1).astype(np.float32)
                tensor = torch.from_numpy(pts9).unsqueeze(0).to(device)

                with torch.no_grad():
                    with torch.cuda.amp.autocast():
                        logits = base_m(tensor)         # (1, n, 2)
                    probs = F.softmax(logits[0],-1)[:,1].cpu().numpy()  # P(ground)

                # Accumulate (expand back to full tile via nearest-repeat)
                if N > max_pts:
                    tile_probs = np.full(N, -1., dtype=np.float64)
                    tile_probs[idx] = probs
                    # For unsampled points: use nearest sampled neighbour prob
                    unsampled = tile_probs < 0
                    if unsampled.any():
                        from scipy.spatial import cKDTree
                        tree = cKDTree(sub_xyz)
                        nn_idx = tree.query(xyz[unsampled], k=1, workers=-1)[1]
                        tile_probs[unsampled] = probs[nn_idx]
                    avg_probs += tile_probs
                else:
                    avg_probs += probs

            avg_probs /= tta_passes    # average over TTA passes

            # Confidence = distance from 0.5 (0=uncertain, 0.5=perfectly confident)
            confidence = np.abs(avg_probs - 0.5) * 2.0  # maps [0.5,1] -> [0,1]
            model_labels = (avg_probs >= 0.5).astype(np.int8)

            # Refined labels: model where confident, CSF where uncertain
            refined = orig_labels.copy()
            high_conf = confidence >= conf_thresh
            refined[high_conf] = model_labels[high_conf]
            n_replaced += high_conf.sum()
            n_total_pts += N

            np.save(str(out_dir/f'{tile}.npy'), refined.astype(np.int8))

    finally:
        if zf_h is not None: zf_h.close()

    pct = 100*n_replaced/max(n_total_pts,1)
    print(f'  Replaced {pct:.1f}% of labels with model predictions (conf>={conf_thresh})')
    return str(out_dir)


# ── Run iterative refinement rounds ──────────────────────────────────────────
from torch.optim.swa_utils import AveragedModel

REFINE_ROUNDS = len(CFG['refine_conf'])

for round_idx in range(REFINE_ROUNDS):
    conf_thr = CFG['refine_conf'][round_idx]
    n_ep     = CFG['refine_epochs'][round_idx]
    lr_r     = CFG['refine_lr']
    round_nm = f'Round{round_idx+1}'
    is_last  = (round_idx == REFINE_ROUNDS-1)

    print(f'\n{"#"*60}')
    print(f'  ITERATIVE REFINEMENT {round_nm}')
    print(f'  Confidence threshold : {conf_thr}')
    print(f'  TTA passes           : {CFG["refine_tta"]}')
    print(f'  Fine-tune epochs     : {n_ep}')
    print(f'  SWA enabled          : {is_last}')
    print(f'{"#"*60}')

    # Step 1: Generate refined labels
    refined_lbl_dir = refine_labels(
        model=model, cfg=CFG, device=DEVICE,
        conf_thresh=conf_thr, tta_passes=CFG['refine_tta'],
        round_name=round_nm,
    )

    # Step 2: Load best weights so far before retraining
    _base = model.module if hasattr(model,'module') else model
    if Path(CFG['model_save_path']).exists():
        _base.load_state_dict(
            torch.load(CFG['model_save_path'],map_location=DEVICE))
        print('Loaded best weights for fine-tuning')

    phase_name = f'Phase{round_idx+2}'
    hist_r, best_r = train_phase(
        cfg        = CFG,
        model      = model,
        phase_name = phase_name,
        n_epochs   = n_ep,
        lr         = lr_r * (0.7**round_idx),  # decay LR each round
        label_dir  = refined_lbl_dir,
        use_swa    = is_last,
        resume_ckpt= None,
    )
    ALL_HISTORY.extend(hist_r)

    print(f'{round_nm} complete: best_acc={best_r*100:.2f}%')

    if (time.time()-SESSION_START)/3600 >= SESSION_BUDGET_H:
        print('Session budget reached — stopping refinement early'); break

# Final curves
plot_all_history(ALL_HISTORY, CFG['curves_path'])

final_best = max(h.get('va',0.) for h in ALL_HISTORY if not math.isnan(h.get('va',float('nan'))))
print(f'\n{"="*60}')
print(f'  ALL PHASES COMPLETE')
print(f'  Best val accuracy across all phases: {final_best*100:.2f}%')
if final_best >= 0.95:
    print('  TARGET >95% : HIT')
else:
    gap=(0.95-final_best)*100
    print(f'  TARGET >95% : {gap:.1f}pp away')
    print("  Add one more round: CFG['refine_conf'].append(0.95)")
    print("                      CFG['refine_epochs'].append(25)")
print(f'{"="*60}')


In [ ]:
# Cell 9 — F1-optimal threshold sweep + package all outputs
import json, numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt, zipfile
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader


def sweep_model(model_path, cfg, device, label='model'):
    if not Path(model_path).exists(): return None
    m=DTMPointNet2(IN_FEAT).to(device)
    m.load_state_dict(torch.load(model_path,map_location=device))
    m.eval()
    va_ld=make_loader(cfg,'val',augment=False)
    all_p,all_l=[],[]
    with torch.no_grad():
        for pts,lbs in tqdm(va_ld,desc=f'Sweep {label}',leave=False):
            with torch.cuda.amp.autocast():
                out=m(pts.to(device))
            all_p.append(F.softmax(out,-1)[:,:,1].view(-1).cpu().numpy())
            all_l.append(lbs.view(-1).numpy())
    probs=np.concatenate(all_p); labs=np.concatenate(all_l)
    best_thr=0.5; best_f1=0.; best_acc=0.; curve=[]
    for thr in np.arange(0.10,0.90,0.01):
        p=(probs>=thr).astype(np.int64)
        tp=((p==1)&(labs==1)).sum(); fp=((p==1)&(labs==0)).sum()
        fn=((p==0)&(labs==1)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
        f1=2*pr*rc/(pr+rc+1e-9); acc=(p==labs).mean()
        curve.append((float(thr),float(f1),float(acc)))
        if f1>best_f1: best_f1=f1; best_thr=float(thr); best_acc=float(acc)
    return dict(model_path=model_path,threshold=best_thr,
                val_accuracy=best_acc,val_f1=best_f1,curve=curve)


results={}
for lbl,path in [('best_epoch',CFG['model_save_path']),('swa',CFG['swa_save_path'])]:
    r=sweep_model(path,CFG,DEVICE,lbl)
    if r:
        results[lbl]=r
        print(f'[{lbl}]  thr={r["threshold"]:.2f}  '
              f'acc={r["val_accuracy"]*100:.2f}%  f1={r["val_f1"]:.4f}')

if not results:
    print('No model files found. Run Cells 7+8 first.'); raise SystemExit()

winner=max(results,key=lambda k:results[k]['val_accuracy'])
best=results[winner]
print(f'\nWinner: {winner}  acc={best["val_accuracy"]*100:.2f}%')

c=best['curve']
fig,ax=plt.subplots(figsize=(8,4))
ax.plot([x[0] for x in c],[x[1] for x in c],label='F1')
ax.plot([x[0] for x in c],[x[2] for x in c],label='Accuracy')
ax.axvline(best['threshold'],ls='--',c='red',label=f'thr={best["threshold"]:.2f}')
ax.axhline(0.95,ls=':',c='orange',alpha=0.8,label='95% target')
ax.set(xlabel='Threshold',ylim=(0.7,1.0),title='Threshold sweep (val set)')
ax.legend(); plt.tight_layout()
plt.savefig(str(Path(CFG['logs_dir'])/'threshold_curve.png'),dpi=120); plt.close()

meta=dict(model=winner,model_path=best['model_path'],
          threshold=best['threshold'],val_accuracy=best['val_accuracy'],
          val_f1=best['val_f1'])
with open(CFG['threshold_path'],'w') as f: json.dump(meta,f,indent=2)

# Package
files=[best['model_path'],CFG['model_save_path'],CFG['swa_save_path'],
       CFG['threshold_path'],CFG['history_path'],CFG['curves_path'],
       str(Path(CFG['logs_dir'])/'threshold_curve.png')]

out_zip=Path('/kaggle/working/dtm_outputs.zip')
with zipfile.ZipFile(str(out_zip),'w',zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if Path(p).exists():
            zf.write(p,Path(p).name)
            print(f'  packed {Path(p).name} ({Path(p).stat().st_size/1e3:.0f} KB)')

print(f'\ndtm_outputs.zip: {out_zip.stat().st_size/1e6:.1f} MB')
print('\nNEXT STEPS:')
print('  Kaggle sidebar -> Output -> dtm_outputs.zip -> Download')
print('  Copy to Terra Pravah backend/models/')

print(f'\nFINAL RESULT')
print(f'  Model     : {winner}')
print(f'  Threshold : {best["threshold"]:.2f}')
print(f'  Accuracy  : {best["val_accuracy"]*100:.2f}%')
if best['val_accuracy']>=0.95: print('  TARGET >95% : HIT')
else: print(f'  TARGET >95% : {(0.95-best["val_accuracy"])*100:.1f}pp away')
